## 7. Experimental Design <a name='design'></a>

### Mathematical Formulation

We model estimated owners as a regression problem. Given a feature vector:

$$\mathbf{x} = [\text{Price},\ \text{Genre},\ \text{Is\_Holiday},\ \text{Positive\_Ratio},\ \text{DLC\_Count},\ \text{Release\_Month}]$$

We seek a function $f$ such that:

$$\hat{y} = f(\mathbf{x}) \approx \log_1(\text{Estimated\_Owners})$$

The optimisation objective is to minimise RMSE over the test set:

$$Z = \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

### Price Optimisation Sub-problem

After training, we use each model to solve:

$$\text{Price}^* = \arg\max_{p \in [0, 70]} f([p,\ \bar{x}_{-\text{price}}])$$

Where $\bar{x}_{-\text{price}}$ is the mean value of all other features. This gives the price point that the model predicts maximises owners.

### Algorithms

| Algorithm | Justification |
|-----------|--------------|
| **Linear Regression** | Baseline. Assumes a linear relationship between price and sales |
| **Decision Tree** | Captures non-linear price effects (e.g. sweet spots at common price points) |
| **Random Forest** | Ensemble — reduces variance, expected to generalise best |

### Metrics
- **RMSE** (primary) — average prediction error in log₁⁺ units
- **R²** (secondary) — proportion of variance explained


## 8. Model Training & Evaluation <a name='models'></a>

In [ ]:
def get_X_y(df):
    feats = [c for c in FEATURE_COLS if c in df.columns]
    return df[feats].select_dtypes(include=[np.number]).copy(), df[TARGET_COL].copy()

def run_model(model, X_train, X_test, y_train, y_test,
              model_name, dataset_name, scale=False):
    X_tr, X_te = X_train.copy(), X_test.copy()
    if scale:
        scaler = StandardScaler()
        X_tr   = scaler.fit_transform(X_tr)
        X_te   = scaler.transform(X_te)
    else:
        scaler = None

    model.fit(X_tr, y_train)

    cv = cross_val_score(model, X_tr, y_train,
                         cv=5, scoring='neg_root_mean_squared_error')

    y_pred = model.predict(X_te)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)

    print(f"    CV RMSE: {-cv.mean():.4f} ± {cv.std():.4f}  |  "
          f"Test RMSE: {rmse:.4f}  |  R²: {r2:.4f}")

    return {
        'Model': model_name, 'Dataset': dataset_name,
        'CV_RMSE': round(-cv.mean(), 4), 'CV_std': round(cv.std(), 4),
        'RMSE': round(rmse, 4), 'R2': round(r2, 4),
        'y_test': y_test.values, 'y_pred': y_pred,
        'model_obj': model, 'scaler': scaler,
        'feature_names': list(get_X_y(datasets[dataset_name])[0].columns),
        'X_train_cols': list(X_train.columns),
        'X_train_means': X_train.mean().to_dict(),
    }

all_results = []

### 8.1 Linear Regression — Baseline

In [ ]:
print("LINEAR REGRESSION")
print("="*60)
for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE)
    r = run_model(LinearRegression(), Xtr, Xte, ytr, yte,
                  'Linear Regression', name, scale=True)
    all_results.append(r)

### 8.2 Decision Tree Regressor

In [ ]:
print("DECISION TREE")
print("="*60)
for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE)
    r = run_model(DecisionTreeRegressor(max_depth=10, min_samples_leaf=5,
                                        random_state=RANDOM_STATE),
                  Xtr, Xte, ytr, yte, 'Decision Tree', name)
    all_results.append(r)

### 8.3 Random Forest Regressor

In [ ]:
print("RANDOM FOREST")
print("="*60)
for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE)
    r = run_model(RandomForestRegressor(n_estimators=100, max_depth=15,
                                        min_samples_leaf=3, n_jobs=-1,
                                        random_state=RANDOM_STATE),
                  Xtr, Xte, ytr, yte, 'Random Forest', name)
    all_results.append(r)

## 9. Sensitivity Analysis <a name='sensitivity'></a>

We examine how key hyperparameters affect model performance to justify our chosen values.


In [ ]:
# Use Dataset 2 (largest) for sensitivity analysis
df_sens = datasets['Dataset 2 (fronkongames)']
X_s, y_s = get_X_y(df_sens)
Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(X_s, y_s,
                                                test_size=TEST_SIZE,
                                                random_state=RANDOM_STATE)

# ── Decision Tree: max_depth ──────────────────────────────────────────────────
depths = [2, 4, 6, 8, 10, 12, 15, 20, None]
tr_rmse, te_rmse = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    m.fit(Xtr_s, ytr_s)
    tr_rmse.append(np.sqrt(mean_squared_error(ytr_s, m.predict(Xtr_s))))
    te_rmse.append(np.sqrt(mean_squared_error(yte_s, m.predict(Xte_s))))

dlabels = [str(d) if d else 'None' for d in depths]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Sensitivity Analysis — Dataset 2', fontweight='bold')

ax = axes[0]
ax.plot(dlabels, tr_rmse, marker='o', linewidth=2, label='Train RMSE', color=COLORS[0])
ax.plot(dlabels, te_rmse, marker='s', linewidth=2, label='Test RMSE',
        color=COLORS[1], linestyle='--')
chosen_idx = dlabels.index('10')
ax.axvline(x=chosen_idx, color='gray', linestyle=':', alpha=0.8)
ax.annotate('Chosen (10)', xy=(chosen_idx, te_rmse[chosen_idx]),
            xytext=(chosen_idx + 0.4, te_rmse[chosen_idx] + 0.02),
            fontsize=8, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_title('Decision Tree: max_depth vs RMSE', fontweight='bold')
ax.set_xlabel('max_depth'); ax.set_ylabel('RMSE (log₁⁺)')
ax.legend(); ax.grid(True, alpha=0.35)

# ── Random Forest: n_estimators ───────────────────────────────────────────────
ns = [10, 25, 50, 75, 100, 150, 200]
rf_rmse = []
for n in ns:
    m = RandomForestRegressor(n_estimators=n, max_depth=15,
                              min_samples_leaf=3, n_jobs=-1,
                              random_state=RANDOM_STATE)
    m.fit(Xtr_s, ytr_s)
    rf_rmse.append(np.sqrt(mean_squared_error(yte_s, m.predict(Xte_s))))

ax = axes[1]
ax.plot(ns, rf_rmse, marker='o', linewidth=2, color=COLORS[2])
ax.axvline(x=100, color='gray', linestyle=':', alpha=0.8, label='Chosen (100)')
ax.set_title('Random Forest: n_estimators vs Test RMSE', fontweight='bold')
ax.set_xlabel('n_estimators'); ax.set_ylabel('Test RMSE (log₁⁺)')
ax.legend(); ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sensitivity.png'), dpi=150)
plt.show()

## 10. Price Optimisation <a name='optimisation'></a>

Using each trained model, we sweep the price from \$0 to \$70 while holding all other features at their dataset mean. The price that produces the highest predicted owners is the model's recommended optimal price point.

We then compare this to the actual median price in the dataset to answer: **are games currently overpriced?**


In [ ]:
price_range = np.linspace(0, 70, 200)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Price Optimisation — Predicted Owners vs Price',
             fontsize=14, fontweight='bold')

opt_results = []

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_xlabel('Price (USD)')
    ax.set_ylabel('Predicted log₁⁺(Owners)')

    actual_median_price = df[df['Price'] > 0]['Price'].median()

    for res in all_results:
        if res['Dataset'] != name:
            continue

        model  = res['model_obj']
        scaler = res['scaler']
        means  = res['X_train_means']
        cols   = res['X_train_cols']

        # Build a synthetic row at mean values, sweeping only price
        predictions = []
        for p in price_range:
            row = {c: means.get(c, 0) for c in cols}
            if 'Price' in row:
                row['Price'] = p
            X_test_row = pd.DataFrame([row])[cols]

            if scaler is not None:
                X_test_row = scaler.transform(X_test_row)

            predictions.append(model.predict(X_test_row)[0])

        predictions = np.array(predictions)
        optimal_idx   = np.argmax(predictions)
        optimal_price = price_range[optimal_idx]

        style = '-' if res['Model'] == 'Random Forest' else '--'
        lw    = 2.5 if res['Model'] == 'Random Forest' else 1.5
        ax.plot(price_range, predictions,
                label=res['Model'], linestyle=style, linewidth=lw)
        ax.axvline(x=optimal_price, linestyle=':', alpha=0.6)
        ax.annotate(f"${optimal_price:.0f}",
                    xy=(optimal_price, predictions[optimal_idx]),
                    xytext=(optimal_price + 1, predictions[optimal_idx] - 0.1),
                    fontsize=7.5, alpha=0.8)

        opt_results.append({
            'Model': res['Model'],
            'Dataset': name,
            'Optimal_Price': round(optimal_price, 2),
            'Actual_Median_Price': round(actual_median_price, 2),
            'Overpriced': actual_median_price > optimal_price,
        })

    # Mark actual median price
    ax.axvline(x=actual_median_price, color='red', linewidth=2,
               linestyle='-', label=f'Actual median (${actual_median_price:.0f})')
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'price_optimisation.png'), dpi=150)
plt.show()

In [ ]:
opt_df = pd.DataFrame(opt_results)

print("Price Optimisation Results")
print("="*65)
display(opt_df)

print("\nSummary by Model:")
summary_opt = (opt_df.groupby('Model')
                     .agg(Avg_Optimal_Price=('Optimal_Price', 'mean'),
                          Avg_Actual_Price=('Actual_Median_Price', 'mean'))
                     .round(2))
summary_opt['Overpriced_By'] = (summary_opt['Avg_Actual_Price']
                                - summary_opt['Avg_Optimal_Price']).round(2)
summary_opt['Verdict'] = summary_opt['Overpriced_By'].apply(
    lambda x: f'Overpriced by ${x:.2f}' if x > 0
              else f'Underpriced by ${abs(x):.2f}')
display(summary_opt)

## 11. Results Summary <a name='results'></a>

In [ ]:
summary_df = pd.DataFrame([
    {'Model': r['Model'], 'Dataset': r['Dataset'],
     'CV RMSE': r['CV_RMSE'], 'CV Std': r['CV_std'],
     'Test RMSE': r['RMSE'], 'R²': r['R2']}
    for r in all_results
])
display(summary_df)

print("\nRMSE Pivot (lower is better):")
display(summary_df.pivot(index='Model', columns='Dataset', values='Test RMSE').round(4))

print("\nR² Pivot (higher is better):")
display(summary_df.pivot(index='Model', columns='Dataset', values='R²').round(4))

In [ ]:
# Model comparison bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Across All Datasets', fontsize=14, fontweight='bold')

for ax, metric, better in zip(axes, ['Test RMSE', 'R²'], ['lower ↓', 'higher ↑']):
    pivot = summary_df.pivot(index='Dataset', columns='Model', values=metric)
    pivot.plot(kind='bar', ax=ax, color=COLORS, edgecolor='white',
               width=0.7, alpha=0.9)
    ax.set_title(f'{metric}  ({better})', fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Model', fontsize=9)
    ax.grid(True, axis='y', alpha=0.35)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}',
                    (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Actual vs Predicted grid (3x3)
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()
mc   = {'Linear Regression': COLORS[0],
        'Decision Tree': COLORS[1], 'Random Forest': COLORS[2]}

for i, res in enumerate(all_results):
    ax = axes[i]
    ax.scatter(res['y_test'], res['y_pred'],
               alpha=0.3, s=7, color=mc[res['Model']])
    lims = [min(res['y_test'].min(), res['y_pred'].min()),
            max(res['y_test'].max(), res['y_pred'].max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5)
    ax.set_title(f"{res['Model']}\n{res['Dataset']}", fontsize=9, fontweight='bold')
    ax.set_xlabel('Actual log₁⁺(Owners)')
    ax.set_ylabel('Predicted')
    ax.text(0.04, 0.93, f"RMSE {res['RMSE']}\nR² {res['R2']}",
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', fc='white', alpha=0.85))

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Actual vs Predicted — Estimated Owners (log₁⁺)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'actual_vs_predicted.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
tree_res = [r for r in all_results
            if r['Model'] in ('Decision Tree', 'Random Forest')]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, res in enumerate(tree_res):
    ax    = axes[i]
    imp   = res['model_obj'].feature_importances_
    feats = res['feature_names']
    dfi   = (pd.DataFrame({'Feature': feats, 'Importance': imp})
               .sort_values('Importance', ascending=True))
    color = COLORS[1] if res['Model'] == 'Decision Tree' else COLORS[2]
    dfi.plot(kind='barh', x='Feature', y='Importance',
             ax=ax, legend=False, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f"{res['Model']}\n{res['Dataset']}", fontweight='bold', fontsize=9)
    ax.set_xlabel('Importance')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Importance — Tree-Based Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'feature_importance.png'), dpi=150)
plt.show()

# How important is Price specifically?
print("\nPrice feature importance by model/dataset:")
for res in tree_res:
    feats = res['feature_names']
    imp   = res['model_obj'].feature_importances_
    if 'Price' in feats:
        idx   = feats.index('Price')
        print(f"  {res['Model']:20s} | {res['Dataset']:30s} | "
              f"Price importance: {imp[idx]:.4f} "
              f"({imp[idx]/imp.sum()*100:.1f}% of total)")

## 12. Discussion & Conclusion <a name='discussion'></a>

> **Complete this section after running all experiments.**  
> Fill in the bracketed placeholders with your actual results.

### 12.1 Which model performed best?
Based on RMSE and R² across all three datasets, [MODEL] consistently achieved the lowest test RMSE of [VALUE], suggesting it generalises best to unseen data. This is consistent with expectations — Random Forest's ensemble averaging reduces the variance that causes single Decision Trees to overfit.

### 12.2 How strongly does price predict sales?
The correlation between price and estimated owners was [VALUE] across datasets. The feature importance plots show that Price accounts for approximately [X]% of the Random Forest's total importance, ranking [Nth] overall. This confirms that price is a meaningful but not dominant predictor — game quality (Positive_Ratio) and genre also play significant roles.

### 12.3 Is there a seasonal effect?
The t-test comparing holiday vs non-holiday releases showed [significant/no significant] difference (p = [VALUE]). Games released in October–December showed [higher/similar] median ownership, consistent with the hypothesis that holiday window releases benefit from increased consumer spending.

### 12.4 What is the optimal price point?
The price optimisation analysis suggests an optimal price of approximately \$[VALUE] (averaged across models and datasets). The current median price in the dataset is \$[ACTUAL].

### 12.5 Are games overpriced?
[Fill in based on opt_df output]  
Based on the model's predictions, games appear to be [overpriced by / underpriced by / priced near] \$[X] relative to the sales-maximising optimum. This suggests that [publishers are pricing above the demand curve's peak / the market could support slightly higher prices].

### 12.6 Limitations
- Steam data is PC-centric and does not represent console pricing dynamics
- Estimated owners is a proxy derived from SteamSpy ranges, not exact sales figures
- The model cannot account for brand recognition, marketing spend, or franchise history
- Price optimisation assumes other features are fixed — in practice, a $70 game may have higher production quality that also increases Positive_Ratio

### 12.7 Stakeholder Takeaway
Our analysis suggests that the sales-maximising price point for the average Steam game is approximately \$[X]. Publishers pricing significantly above this threshold can expect a measurable reduction in player reach. For independent developers especially, pricing at or below \$[X] may dramatically increase their player base while having a modest impact on per-unit revenue.


In [ ]:
# Final summary printout
print("="*65)
print("FINAL RESULTS SUMMARY")
print("="*65)

print("\n── Model Performance ──")
rmse_piv = summary_df.pivot(index='Model', columns='Dataset', values='Test RMSE')
display(rmse_piv.round(4))

print("\n── Price Optimisation ──")
display(opt_df[['Model', 'Dataset',
                'Optimal_Price', 'Actual_Median_Price',
                'Overpriced']].to_string(index=False))

print("\n── Best model per dataset ──")
for col in rmse_piv.columns:
    best = rmse_piv[col].idxmin()
    val  = rmse_piv[col].min()
    print(f"  {col}: {best}  (RMSE = {val:.4f})")

print("\nAll plots saved to:", PLOT_DIR)